In [ ]:
using Plots
using LinearAlgebra
using BenchmarkTools
using Revise
using Jun2Project
using Krylov

# Poisson Solver timing

## Dense Direct

In [ ]:
n_vals = [5, 10, 20, 40, 80];
timing_vals = [];
# errors = zeros(length(n_vals))
for (i, n) in enumerate(n_vals)
    nx = n;
    ny = n;
    Nx = nx-1;
    Ny = ny-1;
    Lx = 1;
    Ly = 1;
    x = LinRange(0, Lx, nx + 1)
    y = LinRange(0, Ly, ny + 1)
    Δx = x[2] - x[1];
    Δy = y[2] - y[1];
    xy = [[x_,y_] for x_ in x, y_ in y]
    xy_interior = [[x_,y_] for x_ in x[2:end-1], y_ in y[2:end-1]]
    L = assemble_laplacian2d_kron(Δx, Δy, Nx, Ny);

    # make the matrix dense
    A = Matrix(-L);
    uex = X-> X[1]*(1-X[1])*X[2]*(1-X[2])
    f = X-> 2*X[2]*(1-X[2])+2* X[1]*(1-X[1]);
    B = vec(f.(xy_interior))
    
    @show n;

    stats = @btimed U = $(A)\$(B);
    push!(timing_vals, stats.time)
    @show stats.time;
    # u = reshape(U, Nx, Ny)
    # errors[i] = norm(u - uex.(xy_interior))
end

In [ ]:
N_vals = @. (n_vals - 1)^2;
scatter(N_vals, timing_vals, xscale=:log10, yscale=:log10, xlabel="N", ylabel="Time (s)", title="Timing of Dense Solve", label="Data Points", legend=:topleft)
plot!(N_vals, 1e-10* N_vals.^3, label="O(N^3)", linestyle=:dash)
plot!(N_vals, 1e-8* N_vals.^2, label="O(N^2)", linestyle=:dash)
xticks!(10 .^ (0:5))
yticks!(10.0 .^ (-6:1))

**Claim** This is a $\mathrm{O}(N^3)$ cost.

## Sparse Direct

In [ ]:
n_vals = [5, 10, 20, 40, 80, 160, 320, 640, 1280];
timing_vals = [];
# errors = zeros(length(n_vals))
for (i, n) in enumerate(n_vals)
    nx = n;
    ny = n;
    Nx = nx-1;
    Ny = ny-1;
    Lx = 1;
    Ly = 1;
    x = LinRange(0, Lx, nx + 1)
    y = LinRange(0, Ly, ny + 1)
    Δx = x[2] - x[1];
    Δy = y[2] - y[1];
    xy = [[x_,y_] for x_ in x, y_ in y]
    xy_interior = [[x_,y_] for x_ in x[2:end-1], y_ in y[2:end-1]]
    L = assemble_laplacian2d_kron(Δx, Δy, Nx, Ny);

    # use the sparse matrix
    A = -L;
    uex = X-> X[1]*(1-X[1])*X[2]*(1-X[2])
    f = X-> 2*X[2]*(1-X[2])+2* X[1]*(1-X[1]);
    B = vec(f.(xy_interior))
    
    @show n;

    stats = @btimed U = $(A)\$(B);
    push!(timing_vals, stats.time)
    @show stats.time;
    # u = reshape(U, Nx, Ny)
    # errors[i] = norm(u - uex.(xy_interior))
end

In [ ]:
N_vals = @. (n_vals - 1)^2;
scatter(N_vals, timing_vals, xscale=:log10, yscale=:log10, xlabel="N", ylabel="Time (s)", title="Timing of Sparse Direct Solve", label="Data Points", legend=:topleft)
# plot!(N_vals, 1e-10* N_vals.^3, label="O(N^3)", linestyle=:dash)
plot!(N_vals, 1e-8* N_vals.^2, label="O(N^2)", linestyle=:dash)
plot!(N_vals, 1e-7* N_vals, label="O(N)", linestyle=:dash)
xticks!(10 .^ (0:5))
yticks!(10.0 .^ (-6:4))

**Claim** This is worse than $\mathrm{O}(N)$, but way better than quadratic, $\mathrm{O}(N^2)$.

## Iterative Solvers
Solution by conjugate gradient, `cg`:

In [ ]:
n_vals = [5, 10, 20, 40, 80, 160, 320, 640, 1280];
timing_vals = [];
# errors = zeros(length(n_vals))
for (i, n) in enumerate(n_vals)
    nx = n;
ny = n;
    Nx = nx-1;
    Ny = ny-1;
    Lx = 1;
    Ly = 1;
    x = LinRange(0, Lx, nx + 1)
    y = LinRange(0, Ly, ny + 1)
    Δx = x[2] - x[1];
    Δy = y[2] - y[1];
    xy = [[x_,y_] for x_ in x, y_ in y]
    xy_interior = [[x_,y_] for x_ in x[2:end-1], y_ in y[2:end-1]]
    L = assemble_laplacian2d_kron(Δx, Δy, Nx, Ny);

    # use the sparse matrix
    A = -L;
    uex = X-> X[1]*(1-X[1])*X[2]*(1-X[2])
    f = X-> 2*X[2]*(1-X[2])+2* X[1]*(1-X[1]);
    B = vec(f.(xy_interior))
    
    @show n;

    stats = @btimed cg($A, $B);
    push!(timing_vals, stats.time)
    @show stats.time;
    # u = reshape(U, Nx, Ny)
    # errors[i] = norm(u - uex.(xy_interior))
end

In [ ]:
N_vals = @. (n_vals - 1)^2;
scatter(N_vals, timing_vals, xscale=:log10, yscale=:log10, xlabel="N", ylabel="Time (s)",
 title="Timing of Sparse Iterative (CG) Solve", label="Data Points", legend=:topleft)
# plot!(N_vals, 1e-10* N_vals.^3, label="O(N^3)", linestyle=:dash)
plot!(N_vals, 1e-8* N_vals.^2, label="O(N^2)", linestyle=:dash)
plot!(N_vals, 1e-7* N_vals, label="O(N)", linestyle=:dash)
xticks!(10 .^ (0:6))
yticks!(10.0 .^ (-6:4))

Better than quadratic, worse than linear.  Why use sparse iterative over sparse direct?

In [ ]:
n = 40;
nx = n;
ny = n;
Nx = nx-1;
Ny = ny-1;
Lx = 1;
Ly = 1;
x = LinRange(0, Lx, nx + 1)
y = LinRange(0, Ly, ny + 1)
Δx = x[2] - x[1];
Δy = y[2] - y[1];
xy = [[x_,y_] for x_ in x, y_ in y]
xy_interior = [[x_,y_] for x_ in x[2:end-1], y_ in y[2:end-1]]
L = assemble_laplacian2d_kron(Δx, Δy, Nx, Ny);

# use the sparse matrix
A = -L;
uex = X-> X[1]*(1-X[1])*X[2]*(1-X[2])
f = X-> 2*X[2]*(1-X[2])+2* X[1]*(1-X[1]);
B = vec(f.(xy_interior))

@show n;
@btime $(A)\$(B);
@btime cg($A, $B);

Sparse iterative is much more memory efficient.

# Heat Equation in 2D